# Pandas 第2课：读写文件与数据选择

> 真实工作流程：**读入数据 -> 选择需要的行列 -> 分析**。
> 本课用一个员工销售表贯穿，先把数据存成 CSV，再练习读取和筛选。
>
> 参考：Pandas 官方「读写表格数据」、w3resource DataFrame 练习。

## 1. 先造一份数据并保存为 CSV

In [1]:
import pandas as pd
import numpy as np

df=pd.DataFrame({
    '工号':[101,102,103,104,105,106],
    '部门':['销售','技术','销售','技术','行政','销售'],
    '姓名':['张三','李四','王五','赵六','钱七','孙八'],
    '销售额':[8000,12000,6500,15000,4000,9500],
    '入职年份':[2019,2020,2021,2018,2022,2020]
})
df.to_csv('员工表.csv', index=False, encoding='utf-8-sig')
print("已保存，Excel打开中文不乱码用 utf-8-sig")

已保存，Excel打开中文不乱码用 utf-8-sig


## 2. read_csv 常用参数

```python
pd.read_csv('文件.csv',
    sep=',',          # 分隔符，制表符用 '\t'
    header=0,         # 第几行作列名，没有列名用 None 并配 names
    names=[...],      # 自定义列名
    usecols=[...],    # 只读某些列（列名或位置）
    nrows=n,          # 只读前 n 行（大文件先看一眼）
    index_col='工号',  # 把某列设为行索引
    dtype={...},      # 指定列类型
    encoding=...)     # 编码，中文常见 utf-8 / gbk
```

In [2]:
df=pd.read_csv('员工表.csv',)
print(df)
small=pd.read_csv('员工表.csv', usecols=['姓名','销售额'], nrows=3)
print(small)

    工号  部门  姓名    销售额  入职年份
0  101  销售  张三   8000  2019
1  102  技术  李四  12000  2020
2  103  销售  王五   6500  2021
3  104  技术  赵六  15000  2018
4  105  行政  钱七   4000  2022
5  106  销售  孙八   9500  2020
   姓名    销售额
0  张三   8000
1  李四  12000
2  王五   6500


### 把某列设为索引（标签定位的前提）

In [3]:
df2=pd.read_csv('员工表.csv', index_col='姓名')
print(df2.index)
print(df2)

Index(['张三', '李四', '王五', '赵六', '钱七', '孙八'], dtype='object', name='姓名')
     工号  部门    销售额  入职年份
姓名                      
张三  101  销售   8000  2019
李四  102  技术  12000  2020
王五  103  销售   6500  2021
赵六  104  技术  15000  2018
钱七  105  行政   4000  2022
孙八  106  销售   9500  2020


## 3. loc 与 iloc（核心，务必分清）

| | loc | iloc |
|---|---|---|
| 全称 | location（标签） | integer location（整数位置） |
| 用什么选 | 行/列的**名字** | 行/列的**数字下标** |
| 切片端点 | **包含**右端 | **不含**右端（和list一样） |

```python
df.loc[行, 列]      # 标签
df.iloc[行, 列]     # 位置
```

In [4]:
df2=pd.read_csv('员工表.csv', index_col='姓名')
#print(df2.loc['李四', '部门'])        # 标签取单个
print(df2.loc['张三':'赵六', '部门':'销售额'])  # 标签切片，含赵六
print(df2.iloc[0:3, 0:3])           # 位置切片，不含第3个

    部门    销售额
姓名           
张三  销售   8000
李四  技术  12000
王五  销售   6500
赵六  技术  15000
     工号  部门    销售额
姓名                
张三  101  销售   8000
李四  102  技术  12000
王五  103  销售   6500


### 用 loc/iloc 做赋值（改特定位置）

In [5]:
d=df2.copy()
d.loc['钱七','销售额']=5000      # 标签定位改值
d.iloc[0,2]=8500               # 位置定位改值
print(d.head(6))

     工号  部门    销售额  入职年份
姓名                      
张三  101  销售   8500  2019
李四  102  技术  12000  2020
王五  103  销售   6500  2021
赵六  104  技术  15000  2018
钱七  105  行政   5000  2022
孙八  106  销售   9500  2020


## 4. 条件筛选（最高频）

### 单条件 / 多条件
- 多条件：`(...) & (...)`、`(...) | (...)`，**括号不能省**
- 多条件不能用 and/or

In [6]:
df=pd.read_csv('员工表.csv')
print(df[df['销售额']>8000])
print(df[(df['部门']=='销售') & (df['销售额']>7000)])
print(df[(df['部门']=='技术') | (df['销售额']>10000)])

    工号  部门  姓名    销售额  入职年份
1  102  技术  李四  12000  2020
3  104  技术  赵六  15000  2018
5  106  销售  孙八   9500  2020
    工号  部门  姓名   销售额  入职年份
0  101  销售  张三  8000  2019
5  106  销售  孙八  9500  2020
    工号  部门  姓名    销售额  入职年份
1  102  技术  李四  12000  2020
3  104  技术  赵六  15000  2018


### isin / between / 字符串筛选

- `isin([...])`：值在清单里
- `between(a,b)`：在区间内（含端点）
- `.str.contains('字')`：字符串包含；`.str.startswith` 等

In [7]:
print(df[df['部门'].isin(['销售','行政'])])
print(df[df['销售额'].between(7000,12000)])
print(df[df['姓名'].str.startswith('张')])

    工号  部门  姓名   销售额  入职年份
0  101  销售  张三  8000  2019
2  103  销售  王五  6500  2021
4  105  行政  钱七  4000  2022
5  106  销售  孙八  9500  2020
    工号  部门  姓名    销售额  入职年份
0  101  销售  张三   8000  2019
1  102  技术  李四  12000  2020
5  106  销售  孙八   9500  2020
    工号  部门  姓名   销售额  入职年份
0  101  销售  张三  8000  2019


### 筛选后只看某些列（loc 组合）

In [8]:
print(df.loc[df['销售额']>8000, ['姓名','销售额']])
print(df[df['销售额']>8000][['姓名','销售额']])

   姓名    销售额
1  李四  12000
3  赵六  15000
5  孙八   9500
   姓名    销售额
1  李四  12000
3  赵六  15000
5  孙八   9500


## 5. 课堂练习

重新读入 `员工表.csv`，完成：
1. 取前 4 行的 `姓名、部门、销售额`
2. 筛选销售额最高的 2 名员工（提示：sort_values+head）
3. 筛选 `2020年及以后入职` 的销售部员工
4. 找出销售额在 6000~10000 之间的员工姓名
5. 把工号设为索引，用 loc 取出 103 号员工的全部信息

In [ ]:
# 在此完成
df3 = pd.read_csv('员工表.csv')
print(df3.iloc[:4][['姓名','部门','销售额']])  # iloc按位置切前4行，再按列名取列（iloc两维都只认数字，不能混用列名）
temp = df3.sort_values(by='销售额', ascending=False)
print(temp.head(2))
print(df3[(df3['入职年份']>=2020) & (df3['部门']=='销售')][['姓名']])
print(df3.loc[(df3['入职年份']>=2020) & (df3['部门']=='销售'), ['姓名']])#第二种写法
print(df3.loc[df3['销售额'].between(6000, 10000), ['姓名']])
print(df3[df3['销售额'].between(6000, 10000)][['姓名']])#第二种写法
print(df3.set_index('工号').loc[103])  # set_index把工号列移到索引，不重复保留


   姓名  部门    销售额
0  张三  销售   8000
1  李四  技术  12000
2  王五  销售   6500
3  赵六  技术  15000
    工号  部门  姓名    销售额  入职年份
3  104  技术  赵六  15000  2018
1  102  技术  李四  12000  2020
   姓名
2  王五
5  孙八
   姓名
2  王五
5  孙八
   姓名
0  张三
2  王五
5  孙八
   姓名
0  张三
2  王五
5  孙八
工号       103
部门        销售
姓名        王五
销售额     6500
入职年份    2021
Name: 103, dtype: object
部门        销售
姓名        王五
销售额     6500
入职年份    2021
Name: 103, dtype: object


### 易错点与注意事项（第2课）

1. **"第几个/前N行"用 iloc；"叫什么名字"用 loc**
   - loc 按标签、切片含端点；iloc 按位置、切片不含端点
   - 默认整数索引时 loc[:3] 和 iloc[:4] 碰巧一样；一旦删过行、索引不连续，loc[:3] 取错行数
   - **iloc 两个维度都只认整数位置，不能行用位置、列用名字**；"前4行+指定列名"用链式：`df.iloc[:4][['姓名','销售额']]`
2. **设索引用 set_index('列')，不要 df.index = df['列']**（后者让该列在索引和普通列里重复）
3. **多条件每个都加括号，用 & / |，不能用 and / or**
4. **df[] 不支持二维 [行,列]，二维定位一律 loc/iloc**
5. **df 在内存、csv 在硬盘**：to_csv 是拍快照（df不变、文件不随后续改动更新），read_csv 是从硬盘加载；index=False 不写行号，中文Excel用 utf-8-sig
6. **布尔索引结果是副本**，改筛选结果不动原表；不想改原表先 .copy()


### 小结

- 读 `read_csv`（usecols/nrows/index_col 常用），写 `to_csv(index=False)`
- loc 按标签、切片含端点；iloc 按位置、切片不含端点
- 条件筛选：`df[条件]`，多条件加括号用 &/|
- isin/between/str.contains 是三大便捷筛选
- 下一课：数据清洗（缺失、重复、类型、字符串）